# Huerto Inteligente — análisis exploratorio

Este cuaderno explora los datos del huerto con **pandas** y **matplotlib**:

1. Cobertura de datos por día (¿cuántas lecturas llegaron de las esperadas?)
2. Calidad: qué lecturas se marcaron y por qué
3. Distribución de cada variable
4. Ciclo diario de temperatura y humedad del aire
5. Curvas de secado entre riegos y su tasa promedio
6. Sensor de la cama frente al clima exterior de la zona
7. Correlaciones

**Datos.** Por defecto lee los datos **sintéticos** de `analisis/datos_sinteticos/`
(se crean con `python tools/generar_sinteticos.py`). Para usar datos reales, exporta con
`python tools/exportar.py --desde AAAA-MM-DD --hasta AAAA-MM-DD` y cambia `RUTA` abajo.
Los datos sintéticos **no son mediciones reales**: sirven para preparar el análisis.

**Limitaciones de los sensores** (detalle en `docs/diccionario_de_datos.md`):
* La temperatura y la humedad del aire vienen de un DHT11: ±2 °C, ±5 puntos de humedad, y entrega enteros.
* El **% de humedad del suelo es una escala calibrada** de la sonda (0 = seca en el aire, 100 = en agua),
  **no** es contenido volumétrico de agua. No se compara directo con el «% vol.» del clima de la zona.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Carpeta con los archivos exportados (CSV o Parquet). Cámbiala para usar datos reales.
RUTA = Path("datos_sinteticos")
ZONA = "America/Santiago"

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
FECHAS = ["medido_en_utc", "recibido_en_utc", "hora_inicio_utc", "inicio_utc", "fin_utc",
          "iniciada_en_utc", "actualizada_en_utc", "resuelta_en_utc", "observado_en_utc"]


def cargar(nombre: str) -> pd.DataFrame:
    """Lee <nombre>.parquet si existe; si no, <nombre>.csv. Fechas en UTC."""
    parquet, csv = RUTA / f"{nombre}.parquet", RUTA / f"{nombre}.csv"
    df = pd.read_parquet(parquet) if parquet.exists() else pd.read_csv(csv, low_memory=False)
    for columna in FECHAS:
        if columna in df.columns:
            df[columna] = pd.to_datetime(df[columna], utc=True)
    if "fecha" in df.columns:
        df["fecha"] = pd.to_datetime(df["fecha"])
    return df


lecturas = cargar("lecturas")
horas = cargar("agregados_hora")
dias = cargar("agregados_dia")
riegos = cargar("eventos_riego")
alertas = cargar("alertas")
clima = cargar("clima")

# Nunca mezclar datos reales con sintéticos
origenes = set().union(*(set(df["origen"].unique()) for df in (lecturas, horas, dias, riegos, alertas, clima)
                         if len(df)))
assert len(origenes) == 1, f"Los archivos mezclan orígenes: {origenes}"
ORIGEN = origenes.pop()
print(f"Origen de los datos: {ORIGEN.upper()}")
lecturas["hora_chile"] = lecturas["medido_en_utc"].dt.tz_convert(ZONA)
print(f"{len(lecturas):,} lecturas de {lecturas['cama'].nunique()} camas, "
      f"del {lecturas['hora_chile'].min():%d-%m-%Y} al {lecturas['hora_chile'].max():%d-%m-%Y}")
lecturas.head()

## 1. Cobertura de datos por día

`cobertura` = lecturas recibidas / lecturas esperadas. La meta del proyecto (CQ-04) es **95 % o más**.
Un día bajo la línea indica cortes de WiFi, de energía o un nodo apagado.

In [ ]:
cobertura = (dias[dias["variable"] == "suelo"]
             .pivot_table(index="fecha", columns="cama", values="cobertura")) * 100
x = np.arange(len(cobertura))
ancho = 0.8 / max(1, cobertura.shape[1])
fig, ax = plt.subplots(figsize=(12, 4))
for k, cama in enumerate(cobertura.columns):
    ax.bar(x + k * ancho, cobertura[cama], width=ancho, label=cama)
ax.axhline(95, color="red", linestyle="--", label="meta 95 %")
ax.set_xticks(x + ancho * (cobertura.shape[1] - 1) / 2, [f"{d:%d-%m}" for d in cobertura.index], rotation=90)
ax.set(ylabel="% de lecturas recibidas", ylim=(0, 105), title="Cobertura de datos por día (humedad del suelo)")
ax.legend(loc="lower right")
plt.show()
cobertura.describe().round(1)

## 2. Calidad de los datos

Cada lectura puede tener marcas: **rango** (fuera de rango físico), **salto** (cambio imposible),
**pegado** (el mismo valor exacto por horas) y **hueco** (faltaron lecturas antes). Las marcadas
rango, salto o pegado se excluyen del análisis (`excluir_*`). La lectura cruda nunca se borra.

In [ ]:
marcas = (lecturas["marcas"].dropna().str.split(";").explode()
          .value_counts().rename_axis("marca").to_frame("lecturas"))
display(marcas)
print("Lecturas excluidas por variable:")
lecturas[["excluir_suelo", "excluir_temp", "excluir_hum_aire"]].sum()

## 3. Distribución de cada variable

Solo valores válidos. Fíjate en la temperatura y la humedad del aire: el DHT11 entrega números
enteros, por eso los histogramas se ven «en escalera».

In [ ]:
VARIABLES = {"suelo_pct": ("Humedad del suelo (% calibrado)", "excluir_suelo"),
             "temp_c": ("Temperatura del aire (°C)", "excluir_temp"),
             "hum_aire_pct": ("Humedad del aire (%)", "excluir_hum_aire")}
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
for eje, (columna, (titulo, excluir)) in zip(ejes, VARIABLES.items()):
    validas = lecturas[lecturas[excluir] == 0]
    for cama, grupo in validas.groupby("cama"):
        eje.hist(grupo[columna].dropna(), bins=40, alpha=0.6, label=cama)
    eje.set_title(titulo)
    eje.legend()
plt.tight_layout()
plt.show()
lecturas.groupby("cama")[list(VARIABLES)].describe().T.round(1)

## 4. Ciclo diario de temperatura y humedad del aire

Promedio por hora del día (hora de Chile). La línea gris es el clima exterior de la zona.

In [ ]:
validas = lecturas[(lecturas["excluir_temp"] == 0) & (lecturas["excluir_hum_aire"] == 0)].copy()
validas["hora_del_dia"] = validas["hora_chile"].dt.hour
ciclo = validas.groupby(["hora_del_dia", "cama"])[["temp_c", "hum_aire_pct"]].mean().unstack("cama")
exterior = clima.assign(hora_del_dia=clima["observado_en_utc"].dt.tz_convert(ZONA).dt.hour)
exterior = exterior.groupby("hora_del_dia")[["temp_aire_c", "hum_rel_pct"]].mean()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4))
ciclo["temp_c"].plot(ax=a1, marker="o")
exterior["temp_aire_c"].plot(ax=a1, color="gray", linestyle="--", label="Afuera (zona)")
a1.set(title="Temperatura promedio por hora", xlabel="Hora del día", ylabel="°C")
a1.legend()
ciclo["hum_aire_pct"].plot(ax=a2, marker="o")
exterior["hum_rel_pct"].plot(ax=a2, color="gray", linestyle="--", label="Afuera (zona)")
a2.set(title="Humedad del aire promedio por hora", xlabel="Hora del día", ylabel="%")
a2.legend()
plt.show()

## 5. Curvas de secado entre riegos

Cada riego detectado inicia un **tramo de secado**, que termina con el riego siguiente. Para cada
tramo se ajusta una recta (regresión lineal) y su pendiente es la **tasa de secado** en puntos por hora.

In [ ]:
suelo = lecturas[(lecturas["excluir_suelo"] == 0) & lecturas["suelo_pct"].notna()]
tramos = []
fig, ejes = plt.subplots(1, suelo["cama"].nunique(), figsize=(14, 4), squeeze=False)
for eje, (cama, serie) in zip(ejes[0], suelo.groupby("cama")):
    eventos = riegos[riegos["cama"] == cama].sort_values("inicio_utc").reset_index(drop=True)
    for i, evento in eventos.iterrows():
        fin = eventos.loc[i + 1, "inicio_utc"] if i + 1 < len(eventos) else serie["medido_en_utc"].max()
        tramo = serie[(serie["medido_en_utc"] > evento["fin_utc"]) & (serie["medido_en_utc"] < fin)]
        if len(tramo) < 12:
            continue
        horas_desde = (tramo["medido_en_utc"] - evento["fin_utc"]).dt.total_seconds() / 3600
        pendiente, _ = np.polyfit(horas_desde, tramo["suelo_pct"], 1)
        tramos.append({"cama": cama, "riego": evento["fin_utc"].tz_convert(ZONA), "duracion_h": horas_desde.max(),
                       "tasa_pts_h": -pendiente, "humedad_inicial": tramo["suelo_pct"].iloc[0]})
        eje.plot(horas_desde, tramo["suelo_pct"], alpha=0.6)
    eje.set(title=f"{cama}: secado después de cada riego", xlabel="Horas desde el riego",
            ylabel="Humedad del suelo (%)")
plt.tight_layout()
plt.show()

tramos = pd.DataFrame(tramos)
print("Tasa de secado promedio (puntos por hora):")
display(tramos.groupby("cama")["tasa_pts_h"].agg(["count", "mean", "std", "min", "max"]).round(2))
tramos.head(10).round({"duracion_h": 1, "tasa_pts_h": 2, "humedad_inicial": 1})

## 6. Sensor de la cama frente al clima exterior

Se compara la **misma hora**: el promedio horario de la cama contra la instantánea horaria de la zona.
Diferencias menores a ±2 °C están dentro del error del DHT11. La humedad del suelo **no** se compara:
la cama usa una escala calibrada y la zona entrega contenido volumétrico de un modelo.

In [ ]:
temp_hora = horas[horas["variable"] == "temp"][["cama", "hora_inicio_utc", "media"]]
clima_hora = clima.assign(hora_inicio_utc=clima["observado_en_utc"].dt.floor("h"))
comparacion = temp_hora.merge(clima_hora[["hora_inicio_utc", "temp_aire_c", "radiacion_wm2"]],
                              on="hora_inicio_utc").dropna()
comparacion["diferencia"] = comparacion["media"] - comparacion["temp_aire_c"]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5))
for cama, g in comparacion.groupby("cama"):
    a1.scatter(g["temp_aire_c"], g["media"], s=8, alpha=0.5, label=cama)
limites = [comparacion["temp_aire_c"].min(), comparacion["temp_aire_c"].max()]
a1.plot(limites, limites, "k--", label="igual que afuera")
a1.fill_between(limites, [l - 2 for l in limites], [l + 2 for l in limites], color="gray", alpha=0.15,
                label="±2 °C (error del DHT11)")
a1.set(xlabel="Temperatura afuera (°C)", ylabel="Temperatura en la cama (°C)", title="Cama vs. zona")
a1.legend()
for cama, g in comparacion.groupby("cama"):
    a2.scatter(g["radiacion_wm2"], g["diferencia"], s=8, alpha=0.5, label=cama)
a2.axhline(0, color="k", linewidth=0.8)
a2.set(xlabel="Radiación solar (W/m²)", ylabel="Cama − afuera (°C)",
       title="¿La cama se calienta más con sol?")
a2.legend()
plt.tight_layout()
plt.show()
comparacion.groupby("cama")["diferencia"].describe().round(2)

## 7. Correlaciones

Correlación de Pearson entre variables horarias. Recuerda: **correlación no es causalidad**.

In [ ]:
ancho = horas.pivot_table(index=["cama", "hora_inicio_utc"], columns="variable", values="media").reset_index()
ancho = ancho.merge(clima_hora[["hora_inicio_utc", "temp_aire_c", "hum_rel_pct", "radiacion_wm2", "et0_mm"]],
                    on="hora_inicio_utc", how="left")
ancho = ancho.rename(columns={"suelo": "suelo_cama", "temp": "temp_cama", "hum_aire": "hum_aire_cama",
                              "temp_aire_c": "temp_afuera", "hum_rel_pct": "hum_afuera",
                              "radiacion_wm2": "radiacion", "et0_mm": "et0"})
# Cambio de la humedad del suelo en la hora (negativo = se seca)
ancho["cambio_suelo"] = ancho.groupby("cama")["suelo_cama"].diff()
columnas = ["suelo_cama", "cambio_suelo", "temp_cama", "hum_aire_cama", "temp_afuera", "hum_afuera",
            "radiacion", "et0"]
matriz = ancho[columnas].corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
imagen = ax.imshow(matriz, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(columnas)), columnas, rotation=45, ha="right")
ax.set_yticks(range(len(columnas)), columnas)
for i in range(len(columnas)):
    for j in range(len(columnas)):
        ax.text(j, i, f"{matriz.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(imagen, ax=ax, shrink=0.8)
ax.set_title("Correlaciones (datos por hora)")
ax.grid(False)
plt.tight_layout()
plt.show()

## Preguntas para la clase

* ¿En qué días la cobertura bajó de 95 %? ¿Qué pudo pasar ese día?
* ¿Por qué la Cama A se seca más rápido que la Cama B? (pista: el sol)
* ¿A qué hora del día la cama está más caliente que afuera? ¿Por qué?
* La tasa de secado, ¿es la misma en todos los tramos? ¿De qué depende?
* ¿Qué variable se relaciona más con el cambio de humedad del suelo? ¿Eso prueba que la causa?